In [ ]:
# Chit 33: Energy consumption anomaly detection
# Step 1: Load and preprocess energy data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense

data = pd.read_csv("energy.csv")
X = data.select_dtypes(include=np.number).dropna().to_numpy(dtype="float32")
X_train, X_test = train_test_split(X, test_size=0.2, shuffle=False)
X_train, X_val = train_test_split(X_train, test_size=0.2)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
# Step 2: Build and train the autoencoder
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(32, activation="relu"),
    Dense(X_train.shape[1], activation="linear")
])
model.compile(optimizer="adam", loss="mse")
model.fit(X_train, X_train, epochs=20, batch_size=32, verbose=0)

In [ ]:
# Step 3: Detect anomalies and display results
val_errors = np.mean(np.square(X_val - model.predict(X_val, verbose=0)), axis=1)
threshold = np.percentile(val_errors, 95)
reconstructed = model.predict(X_test, verbose=0)
errors = np.mean(np.square(X_test - reconstructed), axis=1)
anomalies = errors > threshold
actual = scaler.inverse_transform(X_test)
reconstructed_actual = scaler.inverse_transform(reconstructed)
print("Detected anomalies:", np.where(anomalies)[0])

plt.plot(actual[:, 0], label="Actual")
plt.plot(reconstructed_actual[:, 0], label="Reconstructed")
plt.scatter(np.where(anomalies)[0], actual[anomalies, 0], color="red", label="Anomaly")
plt.legend()
plt.show()

plt.plot(errors)
plt.axhline(threshold, color="red", linestyle="--")
plt.show()